# D1-S04: Combine Northbridge's December exports
**Question:** what did each unit earn in December, and where did every number come from?

- Three unit exports (N01, N02, N03). The folder also holds other months, so load **only** the files the manifest names.
- The key is **entity + period + transaction_id**. `account_code` is **text** (0400 must keep its zero).
- Costs are stored **negative**. Only `posted` rows count. Keep reversals' signs; never use `abs()`.
- Run from the course folder root. Write only to `outputs/D1-S04/`.

The YOUR TURN cells are intentionally empty: *Run All* prepares inputs but does not complete the exercise.  
See slides 1–4 and participant guide sections 1–2.

## 1. Load and inspect (supplied)
Run this, then look at the shape, the types and the missing values.

In [ ]:
# Setup: work from the course folder, whichever folder the notebook kernel started in
import os
from pathlib import Path
here = Path.cwd().resolve()
course_root = next((p for p in [here, *here.parents] if (p / "case_pack").is_dir() and (p / "sessions").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("Open the course folder (the one containing case_pack and sessions) in VS Code.")
os.chdir(course_root)
print("Working directory:", course_root)

In [ ]:
from pathlib import Path
import json
import pandas as pd
folder = Path("case_pack/data/clean")
period = "2025-12"
manifest = json.loads((folder / f"input_manifest_{period}.json").read_text())
parts = []
for relative in manifest["transaction_files"]:
    source = folder / relative
    frame = pd.read_csv(source, dtype=str, keep_default_na=False)
    frame["source_file"] = source.name
    frame["source_row"] = range(1, len(frame) + 1)
    parts.append(frame)
rows = pd.concat(parts, ignore_index=True)
mapping = pd.read_csv(folder / "account_mapping.csv", dtype=str)
display(rows.shape, rows.head(), rows.dtypes, rows.eq("").sum())

## 2. Predict before you run (slide 5)
Fill in your predictions **before** writing any transformation code. `source_controls.csv` holds each unit's own control totals.

In [ ]:
# YOUR TURN: write your predictions as numbers
predicted_posted_rows = None      # how many posted rows across the three units?
predicted_rows_after_join = None  # after joining the mapping?
predicted_n01_result = None       # N01 operating result (revenue + direct cost + overhead)

controls = pd.read_csv(folder / "source_controls.csv", dtype={"period": str})
controls[controls["period"] == period]

## 3. Filter → map → aggregate (slides 6–8, core)
Ask Copilot using a specification (D1-S03 template). Require: posted EUR rows only; `amount` converted to a number after filtering (never `abs()`); a **left** join to `mapping` on text `account_code` with `validate="many_to_one"` and `indicator=True`; totals of signed `amount` by period/entity/report_line named `actual_amount`; rows ordered revenue, direct_cost, overhead within each unit. Paste the code below and **read it before running**.

In [ ]:
# YOUR TURN: create `posted`, `joined` and `by_line`
posted = None
joined = None
by_line = None

## 4. Spot checks (slide 9)
Run after your transformation. Every line should print True.

In [ ]:
if by_line is None:
    print("Complete section 3 first; these checks need posted, joined and by_line.")
else:
    print("rows after filter = prediction:", len(posted) == predicted_posted_rows)
    print("join added no rows:           ", len(joined) == len(posted))
    print("join rows = prediction:       ", len(joined) == predicted_rows_after_join)
    print("join kept the money:          ", round(joined["amount"].sum(), 2) == round(posted["amount"].sum(), 2))
    print("every code mapped:            ", (joined["_merge"] == "both").all())
    n01 = by_line.loc[by_line["entity"] == "N01", "actual_amount"].sum()
    print("N01 result = prediction:      ", round(n01, 2) == predicted_n01_result, round(n01, 2))

## 5. Break it on purpose (slide 10, core)
Predict first: what will `validate="many_to_one"` do with a mapping where one code appears twice? Then load the faulty mapping and rerun your join with it. Afterwards, remove `validate=` once to see the *silent* version (count the rows). Finally restore the clean mapping (do not remove the duplicate yourself) and **rerun section 3** so `joined` and `by_line` are rebuilt from the clean mapping before section 6.

In [ ]:
# YOUR TURN: my prediction: ...
faulty_mapping = pd.read_csv("case_pack/data/faulty/F03_duplicate_mapping/account_mapping.csv", dtype=str)
faulty_mapping[faulty_mapping["account_code"].duplicated(keep=False)]
# rerun your merge with faulty_mapping here, then restore: mapping = the clean file

## 6. Save and compare with the answer (slide 11)
Only after your own result exists. Every key must match and every amount must be within EUR 0.01.

In [ ]:
if by_line is None:
    print("Complete section 3 first; these checks need posted, joined and by_line.")
else:
    out = Path("outputs/D1-S04")
    out.mkdir(parents=True, exist_ok=True)
    by_line.to_csv(out / "actuals_by_line.csv", index=False, float_format="%.2f")
    
    checkpoint = pd.read_csv("case_pack/data/clean/checkpoints/actuals_by_line_2025-12.csv", dtype={"period": str})
    compare = checkpoint.merge(by_line, on=["period", "entity", "report_line"], how="outer",
                               suffixes=("_expected", "_mine"), indicator=True)
    compare["difference"] = (compare["actual_amount_mine"] - compare["actual_amount_expected"]).round(2)
    print("all 9 keys match:", len(compare) == 9 and (compare["_merge"] == "both").all())
    print("all amounts within 0.01:", compare["difference"].abs().le(0.01).all())
    print("business row order:", by_line["report_line"].tolist() == ["revenue", "direct_cost", "overhead"] * 3)
    display(compare)

## 7. Explain (slide 12)
In two sentences each: how could a duplicated mapping row change a plausible total? Why do `source_file` and `source_row` matter? Why keep the positive reversal instead of `abs()`?

**Reset:** reopen this notebook and restart the kernel; inputs are never changed. **Behind?** Later sessions use the supplied checkpoint. **Optional:** repeat for `2026-01` with its own manifest.